# In-depth analysis of Point to Price; Observing what makes a expensive and high score wine?

**Quality: what makes a review sound like a good wine?**

1. At the same price ($15–30), which words and phrases separate high-scoring wines (91–100) from lower-scoring ones (80–86)?
2. Are high-score words about complexity and structure (e.g. long finish, layered, age-worthy) and low-score words about simplicity or flaws (e.g. simple, short, green)?
3. Is the quality vocabulary the same for red and white wine, or does "good" mean something different for each?

**Price: what does a higher price change in the description?**

4. At the same score (87–90), which words separate expensive wines from cheap ones?
5. Are the price differences about quality (praise words) or about style (e.g. oak, polished, concentrated)?

**Back to part 3: Schachner and the Argentina/Chile confusion**

6. Do Schachner's "style" words (raw, rubbery, green, hard, jammy…) also appear when other critics describe low-scoring wines?
7. If so, did his "signature" partly reflect the cheaper, lower-scoring wines he reviews, rather than his writing style?

In [1]:
import re
from functools import lru_cache

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import CountVectorizer, ENGLISH_STOP_WORDS
from data_processing import WINE_WORDS

df = pd.read_pickle(r"C:\Data_Projects\Project_3_Data4401\processed_reviews.pkl")


[nltk_data] Error loading wo9rdnet: Package 'wo9rdnet' not found in
[nltk_data]     index


In [5]:
lemmatizer = WordNetLemmatizer()
TOKEN_PATTERN = r"(?u)\b[a-z][a-z'-]+\b"

@lru_cache(maxsize=None)
def lemma(word):
    return lemmatizer.lemmatize(word)

def lemma_tokenizer(text):
    return [lemma(t) for t in re.findall(TOKEN_PATTERN, text)]

NEGATIONS = {"not", "no", "nor", "without"}
EDGE_STOPS = (set(ENGLISH_STOP_WORDS) - NEGATIONS) | WINE_WORDS
EDGE_STOPS |= {lemma(w) for w in EDGE_STOPS}

COUNTRY_WORDS = {
    "american", "italian", "spanish", "chilean", "australian", "aussie",
    "argentine", "argentinean", "argentinian", "portuguese", "african",
    "european", "europe", "californian", "oregonian", "tuscan", "sicilian",
    "burgundian", "alsatian", "sardinian", "patagonian", "galician", "catalan",
    "iberian", "iberia", "umbrian", "piedmontese", "provencal", "calabrian",
    "venetian", "lombard", "bordelais", "castilian", "french", "old-world", "new-world",
}
BOILERPLATE = {"varietyname", "drink"} | COUNTRY_WORDS

def get_text(d):
    return (d["description_masked"]
              .str.lower()
              .str.replace(r"['’]s\b", "", regex=True)                       # chile's -> chile
              .str.replace(r"\b(french|american|hungarian)\s+oak\b", "oak", regex=True))

def keep_phrase(phrase):
    words = phrase.split()
    if words[0] in EDGE_STOPS or words[-1] in EDGE_STOPS:
        return False
    if "old world" in phrase or "new world" in phrase:
        return False
    return not any(w in BOILERPLATE or w in place_words for w in words) # will create place words in next cell
